In [2]:
import pandas as pd
from dotenv import load_dotenv
import os
from typing import List, Tuple

from tqdm import tqdm

from concurrent.futures import ThreadPoolExecutor, as_completed

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage

In [3]:
df_final = pd.read_csv('sampled_questions - agg.csv')
df_final_labels = df_final.loc[:, ['question', 'final']]
df_final_labels.head()

,question,final
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1


In [4]:
load_dotenv('/Users/egorsmirnov/Desktop/FAQ_autocollector/.env')
parser = StrOutputParser()

models = ['gpt-oss-20b', 'gpt-oss-120b', 'llama-4-scout', 'llama-4-maverick', 'qwen3-14b', 'qwen3-32b', 'qwen3-235b-a22b']

API_KEY = os.getenv('API_KEY')
url = 'https://bothub.chat/api/v2/openai/v1'

llm = ChatOpenAI(api_key=API_KEY, base_url=url, model='llama-4-maverick')

SYSTEM_PROMPT = """
Ты — классификатор вопросов. Твоя задача: определить, является ли вопрос ИНФОРМАТИВНЫМ или НЕИНФОРМАТИВНЫМ.

Верни строго одно число:
1 — если вопрос информативный
0 — если вопрос неинформативный

Определения:ИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который имеет явную цель получить новую, недостающую информацию. Человек хочет узнать факт, способ действия, местоположение, процедуру, контакт, цену, наличие, правила, инструкции, рекомендации или другую конкретную информацию.  
Признаки:
- вопрос направлен на восполнение знаний
- содержит запрос на факты, инструкции, опыт, советы
- человек действительно хочет что‑то узнать

НЕИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который НЕ направлен на получение новой информации.  
Признаки:
- риторический вопрос
- продолжение диалога без запроса знаний
- выражение эмоций, иронии, шутки
- общеизвестный факт, не требующий ответа
- вопрос, не предполагающий полезного ответа

Примеры ИНФОРМАТИВНЫХ вопросов:
- «Где купить свежие морепродукты рядом с Убудом?»
- «Как записаться на маникюр за 5300 ₽?»
- «Кто может помыть окна в ЖК? Поделитесь контактами.»
- «Где посмотреть кроватку ребёнку после колыбельки?»
- «Есть тут фотограф для семейной фотосессии?»

Примеры НЕИНФОРМАТИВНЫХ вопросов:
- «Я могу написать тебе в лс?»
- «А зачем это строить»
- «Бюрки где?» (если контекст отсутствует)
- «А под что маскировать?»
- «Рей или Каору?»
- «Вам таблетки или травы?»

Формат ответа:
Верни только 0 или 1. Никаких пояснений, текста или комментариев.
"""


messages = [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=str(df_final_labels.iloc[0, 0]))]

llm_response = llm.invoke(messages)
result = parser.invoke(llm_response)


print(result)

1


In [5]:
class LLM_classifier:
    
    def __init__(self, model, API_KEY, SYSTEM_PROMPT, url):
        self.parser = StrOutputParser()
        self.model = model
        self.API_KEY = API_KEY
        self.SYSTEM_PROMPT = SYSTEM_PROMPT
        self.url = url
        
    def classify_question(self, llm, question) -> str:
        message = [SystemMessage(content=self.SYSTEM_PROMPT), HumanMessage(content=str(question))
                  ]
        llm_response = llm.invoke(message)
        return self.parser.invoke(llm_response)
        

    def classify_dataframe(self, df_final_labels, model_name) -> pd.DataFrame:
        result = pd.DataFrame()
        result['question'] = df_final_labels['question']

        llm = ChatOpenAI(api_key=self.API_KEY, base_url=self.url, model=model_name)
        labels = []
        for question in tqdm(df_final_labels['question'], desc=f'{model_name}'):
            label = self.classify_question(llm, question)
            labels.append(label)
            
        result[model_name] = labels
        return result


In [10]:
models = ['gpt-oss-20b', 'gpt-oss-120b', 'llama-4-scout', 'llama-4-maverick', 'qwen3-14b', 'qwen3-32b', 'qwen3-235b-a22b']
API_KEY = os.getenv('API_KEY')
url = 'https://bothub.chat/api/v2/openai/v1'

classifier = LLM_classifier(models, API_KEY, SYSTEM_PROMPT, url)

In [127]:
classifier = LLM_classifier(models, API_KEY, SYSTEM_PROMPT, url)
results_df_llama_4_scout = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'llama-4-scout')

llama-4-scout: 100%|██████████████████████████| 329/329 [13:16<00:00,  2.42s/it]


In [128]:
results_df_llama_4_scout

,question,llama-4-scout
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,0


In [135]:
results_df_llama_4_scout.to_csv('results_df_llama_4_scout.csv', index=False)

In [137]:
results_df_gpt_oss_20b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'gpt-oss-20b')
results_df_gpt_oss_20b

gpt-oss-20b: 100%|████████████████████████████| 329/329 [22:21<00:00,  4.08s/it]


,question,gpt-oss-20b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",1
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),1
325,А для Порчи какой лучше класс?,виновколы?»\n\n-??...\n\n-раз?год- Всово? ......
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,"фик ves (.aji?weak? chłрюшка? ...бах —,疑остоят..."


In [138]:
results_df_gpt_oss_20b.to_csv('results_df_gpt_oss_20b.csv', index=False)

In [141]:
results_df_gpt_oss_120b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'gpt-oss-120b')
results_df_gpt_oss_120b

gpt-oss-120b: 100%|███████████████████████████| 329/329 [25:48<00:00,  4.71s/it]


,question,gpt-oss-120b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,1


In [142]:
results_df_gpt_oss_120b.to_csv('results_df_gpt_oss_120b.csv', index=False)

In [145]:
results_df_llama_4_maverick = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'llama-4-maverick')
results_df_llama_4_maverick

llama-4-maverick: 100%|███████████████████████| 329/329 [18:22<00:00,  3.35s/it]


,question,llama-4-maverick
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,0
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [146]:
results_df_llama_4_maverick.to_csv('results_df_llama_4_maverick.csv', index=False)

In [15]:
results_df_qwen3_14b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-14b')
results_df_qwen3_14b

qwen3-14b: 100%|██████████████████████████████| 329/329 [40:15<00:00,  7.34s/it]


,question,qwen3-14b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,0
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [19]:
results_df_qwen3_14b.to_csv('results_df_qwen3_14b.csv', index=False)

In [21]:
results_df_qwen3_32b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-32b')
results_df_qwen3_32b

qwen3-32b: 100%|████████████████████████████| 329/329 [1:02:20<00:00, 11.37s/it]


,question,qwen3-32b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [22]:
results_df_qwen3_32b.to_csv('results_df_qwen3_32b.csv', index=False)

In [28]:
results_df_qwen3_235_a22b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-235b-a22b')
results_df_qwen3_235_a22b

qwen3-235b-a22b: 100%|████████████████████████| 329/329 [48:12<00:00,  8.79s/it]


,question,qwen3-235b-a22b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",1
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,0


In [29]:
results_df_qwen3_235_a22b.to_csv('results_df_qwen3_235_a22b.csv', index=False)

In [17]:
results_df_gpt_oss_20b = pd.read_csv('results_df_gpt_oss_20b.csv')
results_df_gpt_oss_120b = pd.read_csv('results_df_gpt_oss_120b.csv')

results_df_llama_4_scout = pd.read_csv('results_df_llama_4_scout.csv')
results_df_llama_4_maverick = pd.read_csv('results_df_llama_4_maverick.csv')

results_df_qwen3_14b = pd.read_csv('results_df_qwen3_14b.csv')
results_df_qwen3_32b = pd.read_csv('results_df_qwen3_32b.csv')
results_df_qwen3_235_a22b = pd.read_csv('results_df_qwen3_235_a22b.csv')

df_result_models_labels = pd.concat([results_df_gpt_oss_20b, results_df_gpt_oss_120b.iloc[:, 1], 
                                     results_df_llama_4_scout.iloc[:, 1], results_df_llama_4_maverick.iloc[:, 1],
                                     results_df_qwen3_14b.iloc[:, 1], results_df_qwen3_32b.iloc[:, 1], results_df_qwen3_235_a22b.iloc[:, 1]], 
                                    axis=1)

In [19]:
df_result_models_labels.head()

,question,gpt-oss-20b,gpt-oss-120b,llama-4-scout,llama-4-maverick,qwen3-14b,qwen3-32b,qwen3-235b-a22b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1,1,1,1.0,1,1.0,1
1,"А вы первый раз подавали? Я повторно, просто",1,1,1,0.0,0,1.0,1
2,Что вы тогда переживаете? У вас справка на рук...,0,0,0,0.0,0,0.0,0
3,"Победа, можно личный вопрос на публику? Даже д...",1,0,0,0.0,0,0.0,0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1,1,1,1.0,1,1.0,1


In [21]:
df_result_models_labels.to_csv('df_results_models_labels.csv', index=False)

### F1 scores

In [77]:
from sklearn.metrics import f1_score, classification_report

In [69]:
# data cleaning
def is_label(x: str):
    temp = str(x).strip()
    if len(temp) == 0: return -1
    if temp[0] == '0' : return 0
    if temp[0] == '1' : return 1
    return -1

for col in df_result_models_labels.columns[1:]:
    df_result_models_labels[col] = (df_result_models_labels[col].apply(is_label).astype('Int64'))


In [28]:
#df_final = pd.read_csv('sampled_questions - agg.csv')

y_true = df_final['final'].astype(int)

models_f1 = {}
for model in models:
    y_pred = df_result_models_labels[model].fillna(-1).astype('Int64')
    models_f1[model] = f1_score(y_true, y_pred, average='macro');

In [30]:
for model, score in models_f1.items():
    print(f'{model}: {score}\n')

gpt-oss-20b: 0.32547016763804604

gpt-oss-120b: 0.6438843358589763

llama-4-scout: 0.47214654261938876

llama-4-maverick: 0.5207612456747405

qwen3-14b: 0.7641937672613985

qwen3-32b: 0.5117012480049228

qwen3-235b-a22b: 0.7583687669114805



In [36]:
print(classification_report(y_true, df_result_models_labels['qwen3-14b'].fillna(-1).astype('Int64')))

              precision    recall  f1-score   support

           0       0.89      0.61      0.72       160
           1       0.71      0.93      0.81       169

    accuracy                           0.77       329
   macro avg       0.80      0.77      0.76       329
weighted avg       0.80      0.77      0.77       329



## Context based prompt

In [14]:
SYSTEM_POMPT_CONTEXT = '''Ты — классификатор вопросов. Твоя задача: определить, является ли вопрос ИНФОРМАТИВНЫМ (1) или НЕИНФОРМАТИВНЫМ (0).

Верни строго одно число:
1 — информативный вопрос  
0 — неинформативный вопрос  

ОПРЕДЕЛЕНИЯ:

ИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, цель которого восполнить недостающие знания с помощью общеизвестных фактов. Такой вопрос:
- имеет чёткую цель восполнить недостающие знания по определенной теме фактами;
- запрашивает факты, информацию, опыт, инструкции, правила, контакты, местоположение, цены, процедуры;
- имеет широкий контекст: касается большой группы людей, значимой локации, длительного периода времени или общих знаний;
- ответ на него может быть полезен кому‑то ещё, помимо автора.

НЕИНФОРМАТИВНЫЙ ВОПРОС — это вопрос, который НЕ направлен на получение новой полезной информации. Такой вопрос:
- имеет короткий, узкий или личный контекст;
- неясен или непонятно, о чём речь;
- выражает эмоцию, поддерживает разговор, уточняет детали предыдущего сообщения (дознавательный вопрос);
- риторический, саркастический, шуточный;
- не несёт ценности вне конкретного диалога.

ЭВРИСТИКА:
Задай себе вопрос: «Может ли ответ на этот вопрос быть полезен кому‑то ещё?»  
Если да — скорее всего, вопрос информативный.  
Если нет — скорее всего, неинформативный.

НЕСКОЛЬКО ВОПРОСОВ В СООБЩЕНИИ:
Если хотя бы один из вопросов информативный — ставь класс 1.

Примеры ИНФОРМАТИВНЫХ вопросов

- «Девушки, подскажите магазины, где посмотреть кроватку ребёнку, после колыбельки которая. После 2 х лет”»
- «Я очень хочу сделать маникюр за 1 мил / 5 300 ₽ ну очень , как записаться ?»
- «Девушки, кто-нибудь подал на увеличение пособия 3-7 через госуслуги? Не могу найти там это заявление»
- «девочки,где купить морепродуктов свежих рядом с Убудом?»
- «Доброе утро,скажите,пожалуйста,а есть у нас в жк,кто помыть окна может? Поделитесь контактами,пожалуйста»
- «А раньше трава покачивалась?»
- «Девочки, ОПЦ до декабря был на ремонте, кто-нибудь в курсе что именно там отремонтировали?)))»
- «Девочки!Подскажите,где можно купить типичные плетёные круглые сумки по закупочной цене?может кто находил около 100 -120 тыс?хочу несколько купить на сувениры подругам)И такой же вопрос про кокосовое масло.»
- «Привет, девочки!\nЕсть тут фотограф и оператор? Нужны для семейной фотосессии»
- «А для Порчи какой лучше класс?»
- «Если подавали в мфц котельники ,в видном будут документы ?»
- «Может кто подсказать, как дойти до метро?»
- «Всем привет, недавно приехали с мужем в этот замечательный город и не можем найти рынок с продуктами, может подсказать, где его найти?»

Примеры НЕИНФОРМАТИВНЫХ вопросов

- «Я могу написать тебе в лс?» — касается только двух людей.
- «Вам таблетки или травы?» — дознавательный вопрос.
- «А зачем это строить?» — непонятно, о чём речь, дознавательный вопрос.
- «А вы первый раз подавали? Я повторно, просто» — дознавательный вопрос.
- «Бюрки где?» — неясно, о чём идёт речь.
- «А под что маскировать?» — дознавательный вопрос.
- «А микрот туда запилить нельзя?» — недостаточно контекста («туда» — куда?).
- «Рей или Каору?» — опрос мнения.
- «да нефтепродукты это, вы на заправке никогда не были? бензином краску с одежды никогда не смывали?» — саркастичный вопрос.
- «Ну как вам сегодня погода?» — короткий контекст, информация быстро устаревает.
- «Какже хорошо, да?» — продолжение разговора.
- «А во сколько они работают?» — недостаточно контекста («они» — кто?).
- «Спасибо, а что скажете по отоплению, холодно? Угловая квартира всё‑таки.» — неясно, о каком месте речь.

ФОРМАТ ОТВЕТА:
Верни только одно число: 0 или 1.
Никаких пояснений, текста или комментариев.
'''

In [16]:
classifier = LLM_classifier(models, API_KEY, SYSTEM_POMPT_CONTEXT, url)

In [18]:
context_results_df_gpt_oss_20b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'gpt-oss-20b')
context_results_df_gpt_oss_20b

gpt-oss-20b: 100%|████████████████████████████| 329/329 [15:19<00:00,  2.79s/it]


,question,gpt-oss-20b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",1
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,0


In [22]:
context_results_df_gpt_oss_20b.to_csv('context_results_df_gpt_oss_20b.csv', index=False)

In [24]:
context_results_df_gpt_oss_120b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'gpt-oss-120b')
context_results_df_gpt_oss_120b

gpt-oss-120b: 100%|███████████████████████████| 329/329 [27:33<00:00,  5.03s/it]


,question,gpt-oss-120b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,0
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,0


In [32]:
context_results_df_gpt_oss_120b.to_csv('context_results_df_gpt_oss_20b.csv', index=False)

In [34]:
context_results_df_llama_4_scout = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'llama-4-scout')
context_results_df_llama_4_scout

llama-4-scout: 100%|██████████████████████████| 329/329 [11:29<00:00,  2.10s/it]


,question,llama-4-scout
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",1
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,0


In [37]:
context_results_df_llama_4_scout.to_csv('context_labeling/context_results_df_llama_4_scout.csv', index=False)

In [39]:
context_results_df_llama_4_maverick = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'llama-4-maverick')
context_results_df_llama_4_maverick

llama-4-maverick: 100%|███████████████████████| 329/329 [18:45<00:00,  3.42s/it]


,question,llama-4-maverick
0,"Девушки, кто-нибудь подал на увеличение пособи...",
1,"А вы первый раз подавали? Я повторно, просто",
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",1
327,А под что маскировать?,0


In [40]:
context_results_df_llama_4_maverick.to_csv('context_labeling/context_results_df_llama_4_maverick.csv', index=False)

In [41]:
context_results_df_qwen3_14b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-14b')
context_results_df_qwen3_14b

qwen3-14b: 100%|████████████████████████████| 329/329 [1:21:06<00:00, 14.79s/it]


,question,qwen3-14b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [42]:
context_results_df_qwen3_14b.to_csv('context_labeling/context_results_df_qwen3_14b.csv', index=False)

In [43]:
context_results_df_qwen3_32b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-32b')
context_results_df_qwen3_32b

qwen3-32b: 100%|████████████████████████████| 329/329 [1:11:28<00:00, 13.04s/it]


,question,qwen3-32b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),0
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [44]:
context_results_df_qwen3_32b.to_csv('context_labeling/context_results_df_qwen3_32b.csv', index=False)

In [45]:
context_results_df_qwen3_235b_a22b = classifier.classify_dataframe(df_final_labels.loc[:, ['question']], 'qwen3-235b-a22b')
context_results_df_qwen3_235b_a22b

qwen3-235b-a22b: 100%|██████████████████████| 329/329 [1:17:03<00:00, 14.05s/it]


,question,qwen3-235b-a22b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1
1,"А вы первый раз подавали? Я повторно, просто",0
2,Что вы тогда переживаете? У вас справка на рук...,0
3,"Победа, можно личный вопрос на публику? Даже д...",0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1
...,...,...
324,Вы соблюдаете время бодрствования днём ?),1
325,А для Порчи какой лучше класс?,1
326,"Девочки, Чангу! Кто заберёт шлем?\nразмер S, и...",0
327,А под что маскировать?,0


In [46]:
context_results_df_qwen3_235b_a22b.to_csv('context_labeling/context_results_df_qwen3_235b_a22b.csv', index=False)

In [83]:
context_results_df_gpt_oss_20b = pd.read_csv('context_labeling/context_results_df_gpt_oss_20b.csv')
context_results_df_gpt_oss_120b = pd.read_csv('context_labeling/context_results_df_gpt_oss_120b.csv')

context_results_df_llama_4_scout = pd.read_csv('context_labeling/context_results_df_llama_4_scout.csv')
context_results_df_llama_4_maverick = pd.read_csv('context_labeling/context_results_df_llama_4_maverick.csv')

context_results_df_qwen3_14b = pd.read_csv('context_labeling/context_results_df_qwen3_14b.csv')
context_results_df_qwen3_32b = pd.read_csv('context_labeling/context_results_df_qwen3_32b.csv')
context_results_df_qwen3_235b_a22b = pd.read_csv('context_labeling/context_results_df_qwen3_235b_a22b.csv')

df_context_result_models_labels = pd.concat([context_results_df_gpt_oss_20b, context_results_df_gpt_oss_120b.iloc[:, 1], 
                                     context_results_df_llama_4_scout.iloc[:, 1], context_results_df_llama_4_maverick.iloc[:, 1],
                                     context_results_df_qwen3_14b.iloc[:, 1], context_results_df_qwen3_32b.iloc[:, 1], 
                                     context_results_df_qwen3_235b_a22b.iloc[:, 1]], axis=1)

In [85]:
df_context_result_models_labels.head()

,question,gpt-oss-20b,gpt-oss-120b,llama-4-scout,llama-4-maverick,qwen3-14b,qwen3-32b,qwen3-235b-a22b
0,"Девушки, кто-нибудь подал на увеличение пособи...",1.0,1.0,1.0,NaN,1,1.0,1
1,"А вы первый раз подавали? Я повторно, просто",0.0,0.0,0.0,NaN,0,0.0,0
2,Что вы тогда переживаете? У вас справка на рук...,0.0,0.0,0.0,0.0,0,0.0,0
3,"Победа, можно личный вопрос на публику? Даже д...",1.0,0.0,1.0,0.0,0,0.0,0
4,"Доброе утро,скажите,пожалуйста,а есть у нас в ...",1.0,1.0,1.0,1.0,1,1.0,1


### F1 scores for context

In [87]:
df_context_result_models_labels.to_csv('df_context_result_models_labels.csv', index=False)

In [89]:
for col in df_context_result_models_labels.columns[1:]:
    df_context_result_models_labels[col] = (df_context_result_models_labels[col].apply(is_label).astype('Int64'))

In [91]:
y_true = df_final['final'].astype(int)

models_context_f1 = {}
for model in models:
    y_pred = df_context_result_models_labels[model].fillna(-1).astype('Int64')
    models_context_f1[model] = f1_score(y_true, y_pred, average='macro');

for model, score in models_context_f1.items():
    print(f'{model}: {score}\n')

gpt-oss-20b: 0.4560129136400322

gpt-oss-120b: 0.5203034359660865

llama-4-scout: 0.49895103861950774

llama-4-maverick: 0.5197449104733872

qwen3-14b: 0.8317761353600148

qwen3-32b: 0.5493464600697359

qwen3-235b-a22b: 0.5628586430634211



In [93]:
print(classification_report(y_true, df_result_models_labels['qwen3-14b'].fillna(-1).astype('Int64')))

              precision    recall  f1-score   support

           0       0.87      0.78      0.82       160
           1       0.81      0.89      0.85       169

    accuracy                           0.83       329
   macro avg       0.84      0.83      0.83       329
weighted avg       0.84      0.83      0.83       329

